# 1 Импорт библиотек и функций

## 1.1 Библиотеки

In [1]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\baben_bakg1j1\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\baben_bakg1j1\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## 1.2 Функции

In [3]:
def preprocess_headings(df, text_columns=None, language='russian'):
    
    """
    Предобрабатывает текстовые колонки с заголовками
    
    Parameters:
    df - исходный DataFrame
    text_columns - список колонок для обработки (если None, берет все с 'heading_')
    language - язык для стоп-слов и стемминга
    """
    
    if text_columns is None:
        text_columns = [col for col in df.columns if col.startswith('heading_')]
    
    stop_words = set(stopwords.words(language))
    stemmer = SnowballStemmer(language)
    
    def preprocess_text(text):
        if pd.isna(text):
            return ""
        
        # Приведение к нижнему регистру
        text = text.lower()
        
        # Удаление пунктуации и специальных символов
        text = re.sub(r'[^\w\s]', ' ', text)
        
        # Удаление лишних пробелов
        text = re.sub(r'\s+', ' ', text).strip()
        
        # Токенизация
        tokens = word_tokenize(text)
        
        # Удаление стоп-слов и стемминг
        processed_tokens = []
        for token in tokens:
            if token not in stop_words and len(token) > 2:  # удаляем короткие слова
                stemmed_token = stemmer.stem(token)
                processed_tokens.append(stemmed_token)
        
        return ' '.join(processed_tokens)
    
    df_processed = df.copy()
    
    for col in text_columns:
        if col in df_processed.columns:
            df_processed[col] = df_processed[col].apply(preprocess_text)
    
    return df_processed

# 2 Чтение данных

In [5]:
data = pd.read_csv("../../data/news/3_titles_raw.csv")

In [6]:
data.head(2)

,begin,heading_interfax,heading_vedomosti,heading_kommersant
0,2022-05-01 10:00:00,Володин предложил конфисковывать активы владел...,NaN,В России начинается прием заявлений на новую в...
1,2022-05-01 12:00:00,Мишустин подписал постановление о снижении ста...,Правительство одобрило снижение ставки по льго...,NaN


In [7]:
data.isna().sum() / data.shape[0] * 100

begin                  0.000
heading_interfax      20.720
heading_vedomosti      2.304
heading_kommersant     7.120
dtype: float64

# 3 Обработка

In [9]:
df_processed = preprocess_headings(data)

In [10]:
df_processed.heading_interfax

0       володин предлож конфисковыва актив владельц би...
1       мишустин подписа постановлен снижен ставк льго...
2                                                        
3       вступ сил закон ограничива банк передач сведен...
4                                                        
                              ...                        
6245    инвесткомпан permira собра прода neuraxpharm м...
6246                                                     
6247                                                     
6248                                                     
6249                                                     
Name: heading_interfax, Length: 6250, dtype: object

In [11]:
df_processed.to_csv("../../data/news/3_titles_processed.csv", index=False)